# 04 | MCP 通信过程：从 initialize 到 tools/call

这一阶段只回答一个问题：

> Inspector 点击一次按钮后，Client 和 Server 实际交换了什么消息？

本文暂时拿掉 Inspector 这层 UI，用一个极简 MCP Client 直接发送 JSON-RPC 消息，观察 Server 的真实响应。

阅读时只抓三件事：

1. 谁发给谁。
2. 这是 request、response，还是 notification。
3. 关键字段说明了什么。


## 1. 这次要观察什么

实验使用第三篇的订单 Server：

```text
examples/shop_order_primitives_server.py
```

当前 Server 使用 stdio transport。本文观察的是协议层的 JSON-RPC 消息，不是网络包。

本文沿着一条最短链路展开：

```text
initialize
→ notifications/initialized
→ tools/list
→ tools/call
```

读完后，应当能够：

- 判断 request、response、notification。
- 用 `id` 配对 request 和 response。
- 解释为什么要先 `initialize`，再 `tools/list` / `tools/call`。
- 区分 JSON-RPC 错误、tool 执行错误和业务失败。


先把这条链路理解成人话：

```text
Client: 我是谁，我支持哪个协议版本。
Server: 我是谁，我支持哪些能力。
Client: 初始化完成。
Client: 你有哪些 tools？
Server: 这是 tools 列表。
Client: 调用 get_order，参数是 O-1001。
Server: 这是调用结果。
```


## 2. 准备极简 Client

先导入模块，定位订单 Server 文件。这个 Client 只做三件事：启动 stdio Server、写入一行 JSON-RPC 消息、读取一行响应。


In [ ]:
from __future__ import annotations

import json
import subprocess
import sys
from pathlib import Path
from typing import Any


ROOT = Path.cwd()
SERVER = ROOT / "examples" / "shop_order_primitives_server.py"

print(ROOT)
print(SERVER)

下面的辅助函数把每条消息连同方向和类型打印出来。实现不是本文重点，后面只关注它们展示的协议消息。


In [ ]:
def pretty(message: dict[str, Any]) -> None:
    print(json.dumps(message, ensure_ascii=False, indent=2))


def message_kind(message: dict[str, Any]) -> str:
    if "method" in message and "id" in message:
        return "request"
    if "method" in message:
        return "notification"
    if "error" in message:
        return "error response"
    return "response"


def show(direction: str, message: dict[str, Any]) -> None:
    title = f"{direction} {message_kind(message)}"
    if "method" in message:
        title += f" {message['method']}"
    if "id" in message:
        title += f" id={message['id']}"
    print(f"\n# {title}")
    pretty(message)


def start_server() -> subprocess.Popen[str]:
    return subprocess.Popen(
        [sys.executable, str(SERVER)],
        cwd=ROOT,
        stdin=subprocess.PIPE,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True,
        encoding="utf-8",
    )


def write_message(process: subprocess.Popen[str], message: dict[str, Any]) -> None:
    assert process.stdin is not None
    show("Client -> Server", message)
    process.stdin.write(json.dumps(message, ensure_ascii=False) + "\n")
    process.stdin.flush()


def read_message(process: subprocess.Popen[str]) -> dict[str, Any]:
    assert process.stdout is not None
    line = process.stdout.readline()
    if not line:
        stderr = process.stderr.read() if process.stderr else ""
        raise RuntimeError(f"Server closed stdout unexpectedly.\n{stderr}")
    message = json.loads(line)
    show("Server -> Client", message)
    return message


def request(
    process: subprocess.Popen[str],
    request_id: int,
    method: str,
    params: dict[str, Any] | None = None,
) -> dict[str, Any]:
    message: dict[str, Any] = {
        "jsonrpc": "2.0",
        "id": request_id,
        "method": method,
    }
    if params is not None:
        message["params"] = params
    write_message(process, message)
    return read_message(process)


def notify(
    process: subprocess.Popen[str],
    method: str,
    params: dict[str, Any] | None = None,
) -> None:
    message: dict[str, Any] = {
        "jsonrpc": "2.0",
        "method": method,
    }
    if params is not None:
        message["params"] = params
    write_message(process, message)


def stop_server(process: subprocess.Popen[str]) -> None:
    if process.stdin is not None:
        process.stdin.close()
    try:
        process.wait(timeout=2)
    except subprocess.TimeoutExpired:
        process.terminate()
        process.wait(timeout=2)

## 3. 先认识三种消息

JSON-RPC 里最常见的是三种消息。

| 类型 | 关键字段 | 是否期待响应 | 在 MCP 里的例子 |
| --- | --- | --- | --- |
| request | `jsonrpc`、`id`、`method`、`params` | 是 | `initialize`、`tools/list`、`tools/call` |
| response | `jsonrpc`、`id`、`result` 或 `error` | 否，它本身就是响应 | `initialize` 的结果、`tools/call` 的结果 |
| notification | `jsonrpc`、`method`、`params` | 否 | `notifications/initialized` |

判断方式：

- 有 `method` 且有 `id`：request。
- 有 `result` 或 `error` 且有同一个 `id`：response。
- 有 `method` 但没有 `id`：notification。

`id` 的作用是配对请求和响应。Client 发出 `id = 2` 的 `tools/list`，Server 回来的 `id = 2` 就是这条请求的结果。

![JSON-RPC request、response、notification 对比](assets/04-jsonrpc-message-types.svg)

这张图先建立一个直觉：request/response 要用同一个 `id` 配对；notification 没有 `id`，也不等待响应。


**读输出的顺序**

每段输出先看标题，再看 JSON：

1. 方向：`Client -> Server` 还是 `Server -> Client`。
2. 类型：request、response，还是 notification。
3. `id`：请求和响应是否能配对。
4. `method`：这一步在做什么。
5. `result` / `error`：响应成功还是失败。


## 4. 生命周期实验：从启动到初始化

启动订单 MCP Server，并保存子进程对象 `process`。后续消息都通过它的 stdin/stdout 传递。


In [ ]:
process = start_server()
process.poll() is None

输出 `True`，说明 Server 子进程已启动。


**initialize：确认协议版本、身份和能力**

MCP Client 连接 Server 后，第一步必须发送 `initialize`。

这一节只抓一个重点：

> `initialize` 不是在调用业务能力，而是在确认协议版本、双方身份和 Server 能力。

先运行请求，再看响应里最关键的几个字段。


In [ ]:
initialize_response = request(
    process,
    1,
    "initialize",
    {
        "protocolVersion": "2025-11-25",
        "capabilities": {},
        "clientInfo": {
            "name": "mcp-notebook-client",
            "version": "0.1.0",
        },
    },
)

**对请求的解读**

`initialize` request 主要看：

- `id: 1`：用来和响应配对。
- `protocolVersion`：Client 希望使用的协议版本。
- `capabilities: {}`：这是 Client 的能力声明。这里为空，只表示这个极简 Client 没有额外能力。

**对响应的解读**

Server response 主要看：

- `protocolVersion`：最终使用的协议版本。
- `serverInfo.name`：当前 Server 名称。
- `capabilities`：Server 支持的能力。

`capabilities` 里出现 `prompts`、`resources`、`tools`，就表示这些能力存在。里面的 `false` 只表示附加特性没开启，例如列表变化通知或资源订阅。


**notifications/initialized：通知初始化完成**

`initialize` 成功后，Client 发送 `notifications/initialized`。

**对请求的解读**

它没有 `id`，所以不是 request，而是 notification。

**对响应的解读**

没有响应是正常的。notification 只负责通知状态，不等待 Server 回复。

这一步完成后，Client 才开始 `tools/list`、`tools/call` 等后续操作。


In [ ]:
notify(process, "notifications/initialized")

## 5. 发现并调用 Tool

初始化完成后，Client 才开始发现和使用 Server 暴露的能力。本文只沿着 Tool 这条最短链路继续观察。

**tools/list：发现 Server 暴露了哪些工具**

这一步对应 Inspector 里的 `List Tools`。


In [ ]:
tools_response = request(process, 2, "tools/list")

**对请求的解读**

`tools/list` 用来发现 Server 暴露了哪些 tools。它没有参数，只是在问“有哪些工具”。

**对响应的解读**

响应重点看 `result.tools`：

- `name`：tool 名称。
- `description`：tool 用途。
- `inputSchema`：调用参数。
- `annotations`：只读、幂等、是否有破坏性等提示。

这次返回两个 tools：`get_order` 和 `search_orders`。`get_order.order_id` 要符合 `^O-\d{4}$`，例如 `O-1001`。


**tools/call：调用一个 Tool**

发现 `get_order` 之后，Client 可以调用它。

`tools/call` 的关键参数是：

- `params.name`：调用哪个 tool。
- `params.arguments`：传给 tool 的参数对象。

In [ ]:
tool_call_response = request(
    process,
    3,
    "tools/call",
    {
        "name": "get_order",
        "arguments": {
            "order_id": "O-1001",
        },
    },
)

**对请求的解读**

`tools/call` 执行一个具体 tool：

- `params.name`：要调用的 tool，这里是 `get_order`。
- `params.arguments`：传给 tool 的参数，这里是 `order_id: O-1001`。

**对响应的解读**

响应要分三层看：

- 外层有 `result`：说明 JSON-RPC 请求成功。
- `result.isError: false`：说明 tool 执行成功。
- `structuredContent.result.found: true`：说明业务上查到了订单。

出问题时也按这三层排查：协议层、tool 执行层、业务结果层。

到这里，本文的核心链路已经走完：先协商会话，再发现 Tool，最后调用 Tool。


## 6. 可选练习：把读法迁移到 Resource 和 Prompt

下面不再引入新的通信模型，只把相同的 request/response 读法迁移到另外两类 primitive。若只想理解标题中的核心链路，可以直接跳到小结。

**resources/templates/list 与 resources/read：读取资源**

Tool 表示“执行一次操作”。Resource 表示“读取一份可寻址的上下文”。

先发现 Resource Template：

In [ ]:
resource_templates_response = request(process, 4, "resources/templates/list")

**对请求的解读**

`resources/templates/list` 用来发现参数化资源模板。

**对响应的解读**

响应里出现 `shop://orders/{order_id}`，表示 Server 提供订单详情资源模板。它还不是具体订单。


In [ ]:
resource_read_response = request(
    process,
    5,
    "resources/read",
    {
        "uri": "shop://orders/O-1001",
    },
)

**对请求的解读**

`resources/read` 读取一个具体 URI。这里读取的是：

```text
shop://orders/O-1001
```

**对响应的解读**

响应重点看 `result.contents`：

- `uri`：读取的是哪份资源。
- `mimeType`：内容类型。
- `text`：资源正文。

Resource 是读上下文；Tool 是执行操作。


**prompts/list 与 prompts/get：获取 Prompt**

Prompt 表示一组可复用的任务消息模板。先发现 Server 有哪些 prompts：

In [ ]:
prompts_response = request(process, 6, "prompts/list")

**对请求的解读**

`prompts/list` 用来发现 Server 暴露了哪些 prompt 模板。

**对响应的解读**

这次只有一个 prompt：`analyze_one_order`。它有一个可选参数 `order_id`。


In [ ]:
prompt_get_response = request(
    process,
    7,
    "prompts/get",
    {
        "name": "analyze_one_order",
        "arguments": {
            "order_id": "O-1001",
        },
    },
)

**对请求的解读**

`prompts/get` 获取一个具体 prompt，并传入参数。

**对响应的解读**

响应返回的是 messages，不是 tool result。

`prompts/get` 只取回模板，不会自动调用 `get_order`。


**三类 primitive 的方法映射**

| primitive | 发现列表 | 使用能力 |
| --- | --- | --- |
| Tool | `tools/list` | `tools/call` |
| Resource | `resources/list` 或 `resources/templates/list` | `resources/read` |
| Prompt | `prompts/list` | `prompts/get` |

三类 primitive 虽然方法名不同，消息仍然遵循相同模式：先发现，再使用；request 和 response 继续通过 `id` 配对。


**暂不展开的协议机制**

这些机制先知道位置即可：

- pagination：列表太长时，用 `cursor` 分页。
- cancellation：请求太久时，用 notification 取消。
- `listChanged`：列表变化时，Server 是否主动通知 Client。

本实验的 Server 很小，这些机制暂时不用展开。


## 7. 关闭 Server

实验结束后，关闭 Server 子进程。

In [ ]:
stop_server(process)
process.poll() is not None

输出 `True`，说明 Server 子进程已经退出。实验结束后关闭 stdio Server，可以避免后续重复运行时混淆。

## 8. 小结：用同一套顺序读消息

看到一段 MCP 输出，按这个顺序读：

1. 方向：谁发给谁。
2. 类型：request、response、notification。
3. 配对：`id` 是否对应。
4. 方法：`method` 在做什么。
5. 结果：协议成功、tool 成功、业务成功分别是什么。

本实验最重要的结论：

- `initialize` 完成协议版本、身份和能力协商。
- `notifications/initialized` 是通知，没有响应。
- `tools/list` 发现工具，`tools/call` 调用工具。
- 同样的 request/response 模式也适用于 Resource 和 Prompt。

下一阶段进入 Transport：同样的 JSON-RPC 消息，如何通过 stdio 或 Streamable HTTP 传递。


## 参考

- [MCP Basic Protocol](https://modelcontextprotocol.io/specification/2025-11-25/basic/index)
- [MCP Lifecycle](https://modelcontextprotocol.io/specification/2025-11-25/basic/lifecycle)
- [MCP Pagination](https://modelcontextprotocol.io/specification/2025-11-25/server/utilities/pagination)